In [1]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import sys, os
print("python:", sys.version.split()[0])
print("cwd:", os.getcwd())
print("/content:", sorted(os.listdir("/content"))[:25])
print("drive MyDrive:", os.path.isdir("/content/drive/MyDrive"))
print("cpus:", os.cpu_count())
!df -h /content | tail -1
!free -g | head -2

GPU 0: Tesla T4 (UUID: GPU-b469843e-ef9d-617e-0194-18689c4aa784)
Tesla T4, 15360 MiB, 580.82.07
python: 3.13.15
cwd: /content
/content: ['.config', 'sample_data']
drive MyDrive: False
cpus: 2
overlay         113G   43G   71G  38% /
               total        used        free      shared  buff/cache   available
Mem:              12           0           8           0           3          11


In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip uninstall -qqq flash-linear-attention fla-core
# consolidate torch + torchvision from ONE index: the base image ships a torchvision
# whose compiled ops were built against a different torch (-> "operator torchvision::nms does not exist")
!uv pip install -qqq --force-reinstall --no-deps "torch==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
import sys, torch; _t = ".".join(torch.__version__.split(".")[:2]); _cu = (torch.version.cuda or "0").split(".")[0]; _py = f"cp{sys.version_info[0]}{sys.version_info[1]}"; _abi = str(torch.compiled_with_cxx11_abi()).upper()
!uv pip install -qqq "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0+cu{_cu}torch{_t}cxx11abi{_abi}-{_py}-{_py}-linux_x86_64.whl" || echo "No prebuilt causal_conv1d for torch {_t}, using torch fallback"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
print("install step finished")

In [ ]:
import importlib.metadata as md, torch, sys
for p in ["torch","transformers","trl","tokenizers","unsloth","unsloth_zoo","bitsandbytes","peft","datasets","accelerate","torchao"]:
    try: print(f"{p:14s}", md.version(p))
    except Exception as e: print(p, "MISSING", e)
print("cuda:", torch.cuda.is_available(), "| cc:", torch.cuda.get_device_capability(), "| bf16:", torch.cuda.is_bf16_supported())
import unsloth; print("unsloth", unsloth.__version__)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set")

torch          2.8.0
transformers   5.2.0
trl            0.22.2
tokenizers     0.22.2
unsloth        2026.10.1
unsloth_zoo    2026.10.1
bitsandbytes   0.50.2
peft           0.21.1
datasets       4.8.5
accelerate     1.15.0
torchao        0.18.0
cuda: True | cc: (7, 5) | bf16: True


ImportError: Unsloth: torchvision==0.23.0 claims to match torch==2.8.0, but its compiled operators do not load (RuntimeError: operator torchvision::nms does not exist). Reinstall it with `pip install --force-reinstall --no-deps --no-cache-dir "torchvision==0.23.0"`. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to skip this check.

In [3]:
import torch, torchvision, torchvision.ops, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available(), "| dev", torch.cuda.get_device_name(0))
print("torchvision", torchvision.__version__, "| nms ok:", hasattr(torchvision.ops, "nms"))
assert hasattr(torchvision.ops, "nms"), "torchvision ABI mismatch -> re-run the install cell (pins torch 2.8.0 + torchvision 0.23.0 from the cu128 index)"
import unsloth
print("UNSLOTH IMPORT OK", unsloth.__version__)

python 3.13.15
torch 2.8.0+cu128 | cuda 12.8 | available True | dev Tesla T4
torchvision 0.23.0+cu128 | nms ok: True


/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:110: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.13/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
UNSLOTH IMPORT OK 2026.10.2


In [4]:
# ============ CONFIG (single source of truth) ============
import os, json, time, random
import numpy as np, torch

SEED             = 3407
STUDENT_MODEL    = os.environ.get("STUDENT_MODEL", "unsloth/Qwen3-0.6B")
HF_DATASET       = "p4ulbr4dl3y/ru-contextual-chunks"   # clean 3134-row v2 (chunk in doc = 100%)
MAX_SEQ_LENGTH   = 16384
OUT_DIR          = "/content/out"
# RUN_ROOT / CKPT_DIR / ART_DIR / LOGDIR are resolved by the "DRIVE + RUN DIR" cell (works on a shared account too)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0.0
BS, GRAD_ACC, EVAL_BS = 2, 4, 1      # measured on T4: bs2 x ga4 = ~43 s / (8 samples x 9.3k tok)
LR, EPOCHS, WARMUP   = 2e-4, 2, 5    # epoch 1 done: 338 steps, best eval_loss 0.5119 @ step 320; epoch 2 = continuation run (resumes from checkpoint-338)
SAVE_EVERY           = 40            # eval + checkpoint cadence (steps)
EARLY_STOP_PATIENCE  = 3
VAL_EVAL_N           = 120           # in-training eval subset (full 433 x bs1 is too slow)
METRIC_BS_N          = 60            # val rows for word-F1 generation metrics
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CONFIG_KEYS = ("SEED", "STUDENT_MODEL", "HF_DATASET", "MAX_SEQ_LENGTH", "OUT_DIR", "LORA_R", "LORA_ALPHA",
               "LORA_DROPOUT", "BS", "GRAD_ACC", "EVAL_BS", "LR", "EPOCHS", "WARMUP", "SAVE_EVERY",
               "EARLY_STOP_PATIENCE", "VAL_EVAL_N", "METRIC_BS_N")
print(json.dumps({k: globals()[k] for k in CONFIG_KEYS if k in globals()}, indent=1))

{
 "SEED": 3407,
 "STUDENT_MODEL": "unsloth/Qwen3-0.6B",
 "HF_DATASET": "p4ulbr4dl3y/ru-contextual-chunks",
 "MAX_SEQ_LENGTH": 16384,
 "OUT_DIR": "/content/out",
 "LORA_R": 16,
 "LORA_ALPHA": 16,
 "LORA_DROPOUT": 0.0,
 "BS": 2,
 "GRAD_ACC": 4,
 "EVAL_BS": 1,
 "LR": 0.0002,
 "EPOCHS": 2,
 "WARMUP": 5,
 "SAVE_EVERY": 40,
 "EARLY_STOP_PATIENCE": 3,
 "VAL_EVAL_N": 120,
 "METRIC_BS_N": 60
}


In [5]:
# ============ STUDENT MODEL + LoRA ============
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = STUDENT_MODEL,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    load_in_8bit    = False,
    full_finetuning = False,
)
print("model:", STUDENT_MODEL, "| class:", type(model).__name__)
print("dtype:", next(model.parameters()).dtype, "| device:", next(model.parameters()).device)
print("tokenizer:", type(tokenizer).__name__, "| vocab:", tokenizer.vocab_size,
      "| eos:", repr(tokenizer.eos_token), tokenizer.eos_token_id,
      "| pad:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # causal LM: pad right
print("pad forced to eos:", repr(tokenizer.pad_token), tokenizer.pad_token_id)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,       # 0 keeps unsloth fast patching enabled
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)
model.print_trainable_parameters()
print("peak VRAM after model load: %.2f GB" % (torch.cuda.max_memory_allocated()/1e9))

==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

model: unsloth/Qwen3-0.6B | class: Qwen3ForCausalLM
dtype: torch.float16 | device: cuda:0
tokenizer: Qwen2Tokenizer | vocab: 151643 | eos: '<|im_end|>' 151645 | pad: '<|vision_pad|>' 151654
pad forced to eos: '<|im_end|>' 151645


Unsloth 2026.10.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
peak VRAM after model load: 0.65 GB


In [6]:
# ============ DATA (public v2, verified clean) ============
import re, collections
from datasets import load_dataset

ds = load_dataset(HF_DATASET)
train_ds, val_ds = ds["train"], ds["validation"]
print("splits:", {k: len(v) for k, v in ds.items()}, "| cols:", train_ds.column_names)

WS = re.compile(r"\s+")
def parse_prompt(content):
    doc   = content.split("<document>\n", 1)[1].split("\n</document>", 1)[0]
    chunk = content.split("<chunk>\n", 1)[1].split("\n</chunk>", 1)[0]
    return doc, chunk

def integrity(split, name):
    ok = 0; bad = []
    for ex in split:
        doc, chunk = parse_prompt(ex["messages"][0]["content"])
        if WS.sub(" ", chunk).strip() in WS.sub(" ", doc).strip():
            ok += 1
        else:
            bad.append(ex["id"])
    print(f"{name}: chunk-inside-document {ok}/{len(split)} ({ok/len(split)*100:.1f}%)", "BAD:" + str(bad[:3]) if bad else "")
    return ok == len(split)

assert integrity(train_ds, "train") and integrity(val_ds, "val"), "integrity check failed"

overlap = set(train_ds["window_hash"]) & set(val_ds["window_hash"])
assert not overlap, f"train/val window_hash leak: {len(overlap)}"
print("window_hash overlap train/val:", len(overlap))
print("doc groups train/val:", len(set(train_ds["window_hash"])), "/", len(set(val_ds["window_hash"])))
print("categories:", dict(collections.Counter(train_ds["category"]).most_common()))

# ---- render to chat text (same template for train and inference) ----
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}

train_t = train_ds.map(to_text, desc="render")
val_t   = val_ds.map(to_text, desc="render")

lens = []
for t in train_t["text"]:
    lens.append(len(tokenizer(text=t, add_special_tokens=False)["input_ids"]))
lens_arr = sorted(lens)
n = len(lens_arr)
print(f"train token len: min {lens_arr[0]} | p50 {lens_arr[n//2]} | p95 {lens_arr[int(n*.95)]} | p99 {lens_arr[int(n*.99)]} | max {lens_arr[-1]}")
val_lens = sorted(len(tokenizer(text=t, add_special_tokens=False)["input_ids"]) for t in val_t["text"])
print(f"val   token len: p50 {val_lens[len(val_lens)//2]} | max {val_lens[-1]}")
assert lens_arr[-1] <= MAX_SEQ_LENGTH and val_lens[-1] <= MAX_SEQ_LENGTH, "raise MAX_SEQ_LENGTH"
print("no sample exceeds MAX_SEQ_LENGTH =", MAX_SEQ_LENGTH)
print("=== sample render (tail) ===")
print(repr(train_t[0]["text"][-260:]))

README.md:   0%|          | 0.00/612 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.03MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/433 [00:00<?, ? examples/s]

splits: {'train': 2701, 'validation': 433} | cols: ['id', 'doc_hash', 'category', 'messages', 'window_hash']
train: chunk-inside-document 2701/2701 (100.0%) 
val: chunk-inside-document 433/433 (100.0%) 
window_hash overlap train/val: 0
doc groups train/val: 272 / 48
categories: {'it': 1614, 'food': 377, 'security': 289, 'pharma': 228, 'construction': 193}


render:   0%|          | 0/2701 [00:00<?, ? examples/s]

render:   0%|          | 0/433 [00:00<?, ? examples/s]

train token len: min 439 | p50 8585 | p95 9435 | p99 9481 | max 9728
val   token len: p50 8644 | max 9680
no sample exceeds MAX_SEQ_LENGTH = 16384
=== sample render (tail) ===
'нт открывает контракт № 0321500001626000075 между ГБУ СК «Стававтодор» и ООО «ЭПО «Результат» на оказание услуг по лабораторным испытаниям бетона на прочность и содержит преамбулу, предмет контракта, перечень объектов ремонта и место оказания услуг.<|im_end|>\n'


In [12]:
# ============ GENERATION METRICS (word-F1 vs teacher; defines run_metrics) ============
import json, os, random, re, time, torch
import numpy as np
from unsloth import FastLanguageModel

STOP = {"фрагмент","контракт","документ","который","описывает","содержит","оказание","услуг","услуги",
        "материалов","строительных","рамках","также","данный","этого","всего","всех","которые"}
def words(t):
    return set(x for x in re.findall(r"[а-яёa-z0-9]+", t.lower()) if len(x) >= 5 and x not in STOP)
def f1(pred, ref):
    P, R = words(pred), words(ref)
    if not P or not R: return 0.0
    i = len(P & R); pr, rc = i/len(P), i/len(R)
    return 2*pr*rc/(pr+rc) if pr+rc else 0.0
def numbers(t):
    return set(re.findall(r"\d+", t))

def eval_rows(n=METRIC_BS_N, seed=0):
    """deterministic subsample of the validation split"""
    idx = list(range(len(val_ds))); random.Random(seed).shuffle(idx)
    return [val_ds[i] for i in sorted(idx[:n])]

@torch.inference_mode()
def gen_one(row, max_new_tokens=160):
    content = row["messages"][0]["content"]
    doc, chunk = parse_prompt(content)
    txt = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tokenizer(text=txt, return_tensors="pt").to("cuda")
    t0 = time.time()
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    dt = time.time() - t0
    gen = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return gen, dt, doc + " " + chunk

def run_metrics(tag, rows):
    FastLanguageModel.for_inference(model)
    res, times = [], []
    for row in rows:
        gen, dt, ctx = gen_one(row)
        ref = row["messages"][1]["content"]
        res.append({"id": row["id"], "category": row["category"], "gen": gen, "ref": ref,
                    "f1": f1(gen, ref), "n_new_tokens": len(tokenizer(gen, add_special_tokens=False)["input_ids"]),
                    "halluc_numbers": len(numbers(gen) - numbers(ctx)),
                    "lev": 0})
        times.append(dt)
    # median and p10 are computed over all rows. The median used to be taken as
    # scores[n//2] (the upper element, not the mean of the two middle ones), which
    # biased it upward in every report, in the paper and in the model card; p10 was
    # taken as scores[int(n*.1)] instead of a real percentile.
    scores = sorted(r["f1"] for r in res)
    summary = {
        "tag": tag, "n": len(res),
        "f1_mean": round(sum(scores)/len(scores), 4),
        "f1_median": round(float(np.median(scores)), 4),
        "f1_p10": round(float(np.percentile(scores, 10)), 4),
        "f1_zero_frac": round(sum(1 for s in scores if s == 0)/len(scores), 4),
        "halluc_numbers_total": sum(r["halluc_numbers"] for r in res),
        "gen_sec_per_sample": round(sum(times)/len(times), 2),
        "from": rows[0]["id"][:30],
    }
    by_cat = {}
    for r in res:
        by_cat.setdefault(r["category"], []).append(r["f1"])
    summary["f1_by_category"] = {k: round(sum(v)/len(v), 3) for k, v in sorted(by_cat.items())}
    print(tag, json.dumps(summary, ensure_ascii=False))
    return summary, res

EVAL_ROWS = eval_rows()
print("metric rows:", len(EVAL_ROWS), "| categories:", {k: sum(1 for r in EVAL_ROWS if r['category']==k) for k in sorted(set(r['category'] for r in EVAL_ROWS))})

# The baseline is deterministic (greedy decode, fixed row sample). Measure it once and restore it from HF
# on a fresh runtime (RUN DIR cell) - never overwrite it once training started.
if os.path.exists(f"{OUT_DIR}/metrics_baseline.json"):
    baseline_summary = json.load(open(f"{OUT_DIR}/metrics_baseline.json"))["summary"]
    print("baseline already measured -> skip:", json.dumps(baseline_summary, ensure_ascii=False))
else:
    baseline_summary, baseline_res = run_metrics("BASELINE(untrained)", EVAL_ROWS)
    json.dump({"summary": baseline_summary, "rows": baseline_res},
              open(f"{OUT_DIR}/metrics_baseline.json", "w"), ensure_ascii=False, indent=1)
    print("saved", f"{OUT_DIR}/metrics_baseline.json")


metric rows: 60 | categories: {'construction': 2, 'food': 20, 'it': 34, 'pharma': 2, 'security': 2}
baseline already measured -> skip: {"tag": "BASELINE(untrained)", "n": 60, "f1_mean": 0.1893, "f1_median": 0.186, "f1_p10": 0.0645, "f1_zero_frac": 0.0167, "halluc_numbers_total": 0, "gen_sec_per_sample": 5.55, "from": "construction_62_Плат_жное_пору", "f1_by_category": {"construction": 0.097, "food": 0.143, "it": 0.232, "pharma": 0.137, "security": 0.079}}


In [ ]:
# ============ GOOGLE DRIVE (checkpoints + artifacts), retry with longer timeout ============
from google.colab import drive
drive.mount("/content/drive", force_remount=True, timeout_ms=280000)

import os, glob
print("MyDrive:", sorted(os.listdir("/content/drive/MyDrive"))[:30])
for extra in ["/content/drive/Shareddrives", "/content/drive/Othercomputers"]:
    if os.path.isdir(extra):
        print(extra + ":", sorted(os.listdir(extra))[:20])
print("shortcuts:", sorted(os.path.basename(p) for p in glob.glob(os.path.expanduser("~/.shortcut-targets-by-id/*/*")))[:20])

Mounted at /content/drive
MyDrive: ['Colab Notebooks', 'training']
shortcuts: []


In [8]:
# ============ RUN DIR + DURABLE OFF-SITE BACKUP ============
# Preferred: the shared 'training' folder on Drive (same place as the previous checkpoints).
# Fallback: fast local disk + mirror every checkpoint to a private HF repo.
import os, glob, shutil
from google.colab import drive
from huggingface_hub import HfApi, snapshot_download, hf_hub_download

HF_MIRROR_REPO = "p4ulbr4dl3y/qwen3-0.6b-contextual-chunks-checkpoints"  # private: resume/backup store
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True, timeout_ms=280000)
print("MyDrive:", sorted(os.listdir("/content/drive/MyDrive"))[:12])

cands = []
for pat in ("/content/drive/MyDrive/training", "/content/drive/MyDrive/*training*",
            "/content/drive/Shareddrives/*/training", "/content/drive/Shareddrives/*/*training*",
            "/content/drive/Othercomputers/*/*training*"):
    cands += glob.glob(pat)
cands += glob.glob(os.path.expanduser("~/.shortcut-targets-by-id/*/*"))
cands = [p for p in cands if os.path.isdir(p)]
with_ckpt = [p for p in cands if glob.glob(p + "/contextual_chunks_sft_v2/checkpoints/checkpoint-*")]
with_run  = [p for p in cands if os.path.isdir(p + "/contextual_chunks_sft_v2")]
base = (with_ckpt or with_run or [None])[0]

try: HF_TOKEN = os.environ.get("HF_TOKEN") or __import__("google.colab").userdata.get("HF_TOKEN")
except Exception: HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN: os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

if base and os.access(base, os.W_OK):
    RUN_ROOT = os.path.join(base, "contextual_chunks_sft_v2")
    print("storage: Google Drive |", RUN_ROOT)
else:
    RUN_ROOT = "/content/run"
    print("storage: local disk (/content/run) + HF mirror")
    assert HF_TOKEN, ("training folder not found on Drive: add a My Drive shortcut to the shared 'training' folder, "
                      "or provide an HF token (env HF_TOKEN / Colab secret HF_TOKEN) so checkpoints can be restored from HF")
    files = HfApi().list_repo_files(HF_MIRROR_REPO, repo_type="model")
    steps = sorted({int(p.split("/")[1].rsplit("-", 1)[1]) for p in files
                    if p.startswith("checkpoints/checkpoint-") and p.endswith("/optimizer.pt")})
    print("remote resume points on HF:", steps)
    if steps:
        latest = steps[-1]; dst = f"/content/run/checkpoints/checkpoint-{latest}"
        if not os.path.isdir(dst):
            snap = snapshot_download(HF_MIRROR_REPO, repo_type="model", allow_patterns=f"checkpoints/checkpoint-{latest}/*")
            os.makedirs("/content/run/checkpoints", exist_ok=True)
            shutil.copytree(os.path.join(snap, "checkpoints", f"checkpoint-{latest}"), dst)
        print("restored from HF:", os.path.basename(dst), "|", sorted(os.listdir(dst)))

# baseline metrics for the post-training comparison (deterministic; restore on a fresh runtime)
try:
    if not os.path.exists(f"{OUT_DIR}/metrics_baseline.json"):
        os.makedirs(OUT_DIR, exist_ok=True)
        hf_hub_download(repo_id="p4ulbr4dl3y/qwen3-0.6b-contextual-chunks", filename="metrics_baseline.json",
                        repo_type="model", local_dir=OUT_DIR)
        print("restored baseline metrics from HF")
except Exception as e:
    print("baseline restore skipped:", type(e).__name__, str(e)[:120])

CKPT_DIR, ART_DIR, LOGDIR = f"{RUN_ROOT}/checkpoints", f"{RUN_ROOT}/artifacts", f"{RUN_ROOT}/logs"
for d in (CKPT_DIR, ART_DIR, LOGDIR): os.makedirs(d, exist_ok=True)
print("RUN_ROOT:", RUN_ROOT, "| writable:", os.access(RUN_ROOT, os.W_OK))
print("checkpoints:", sorted(os.path.basename(p) for p in glob.glob(CKPT_DIR + "/checkpoint-*")))
print("artifacts:", sorted(os.listdir(ART_DIR)))
if HF_TOKEN:
    print("hf user:", HfApi().whoami()["name"], "| mirror target:", HF_MIRROR_REPO)
print("disk free: %.1f GB" % (shutil.disk_usage(RUN_ROOT).free / 1e9))

Mounted at /content/drive
MyDrive: ['1.\tОнтология - учение о бытии и сущности сущего.  \t2.\tГносеология - учени~изм - учение, считающее, что все события заранее определены судьбой. .txt', '3 вопроса.gform', '5C7F846E-4366-454C-8DB0-F38F5E020F3B.jpeg', 'Classroom', 'Colab Notebooks', 'Diffusion Models.pptx', 'Marta-14.pdf', 'PP_04_maket_otchyota.doc', 'Record (online-voice-recorder.com).mp3', 'VSCodeUserSetup-x64-1.89.0.exe', 'adult.csv', 'agaricus-lepiota.data']
storage: local disk (/content/run) + HF mirror
remote resume points on HF: [160, 200, 240, 280, 320, 338, 360, 400, 440, 480, 520, 560]


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

restored from HF: checkpoint-560 | ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'optimizer.pt', 'rng_state.pth', 'scaler.pt', 'scheduler.pt', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']


metrics_baseline.json:   0%|          | 0.00/71.4k [00:00<?, ?B/s]

restored baseline metrics from HF
RUN_ROOT: /content/run | writable: True
checkpoints: ['checkpoint-560']
artifacts: []
hf user: p4ulbr4dl3y | mirror target: p4ulbr4dl3y/qwen3-0.6b-contextual-chunks-checkpoints
disk free: 63.9 GB


In [9]:
# ============ TRAINER + MASKING SANITY ============
import os, json, glob, time, torch
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback, TrainerCallback
from huggingface_hub import HfApi
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

# RUN_ROOT / CKPT_DIR / ART_DIR / LOGDIR come from the "RUN DIR" cell above
assert "CKPT_DIR" in globals(), "run the RUN DIR cell first"
for d in (CKPT_DIR, ART_DIR, LOGDIR):
    os.makedirs(d, exist_ok=True)

# --- eval subset: full 433-row validation is too slow for in-training eval (bs=1, ~8.6k tok) ---
VAL_EVAL_N = 120
val_eval = val_t.select(range(VAL_EVAL_N))
print("val for in-training eval:", len(val_eval), "of", len(val_t), "(deterministic prefix, order fixed by split seed)")

class JsonlLogger(TrainerCallback):
    def __init__(self, path): self.path = path
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        rec = {"step": state.global_step, **{k: v for k, v in logs.items() if isinstance(v, (int, float))}}
        with open(self.path, "a") as f: f.write(json.dumps(rec) + "\n")
    def on_save(self, args, state, control, **kw):
        with open(self.path, "a") as f: f.write(json.dumps({"event": "save", "step": state.global_step}) + "\n")

class HFPushCallback(TrainerCallback):
    """Mirror every checkpoint to the private HF repo HF_MIRROR_REPO: survives runtime death / GPU-quota kills."""
    def __init__(self, repo_id, path_in_repo="checkpoints"):
        self.repo_id, self.path_in_repo = repo_id, path_in_repo
    def on_save(self, args, state, control, **kw):
        if not os.environ.get("HF_TOKEN"): return
        ck = f"{args.output_dir}/checkpoint-{state.global_step}"
        if not os.path.isdir(ck): return
        try:
            HfApi().upload_folder(folder_path=ck, repo_id=self.repo_id, repo_type="model",
                                  path_in_repo=f"{self.path_in_repo}/{os.path.basename(ck)}",
                                  commit_message=f"checkpoint step {state.global_step}")
            print(f"[HF] mirrored {os.path.basename(ck)}", flush=True)
        except Exception as e:
            print("[HF] mirror failed:", type(e).__name__, str(e)[:200], flush=True)

def build_trainer(max_steps=-1, eval_steps=SAVE_EVERY, smoke=False):
    cfg = SFTConfig(
        output_dir            = f"{CKPT_DIR}_smoke" if smoke else CKPT_DIR,
        dataset_text_field    = "text",
        max_seq_length        = MAX_SEQ_LENGTH,
        packing               = False,
        per_device_train_batch_size = BS,
        per_device_eval_batch_size  = EVAL_BS,   # unsloth + bs>1 -> eval logits OOM
        gradient_accumulation_steps = GRAD_ACC,
        warmup_steps          = WARMUP,
        num_train_epochs      = EPOCHS,
        max_steps             = max_steps,
        learning_rate         = LR,
        lr_scheduler_type     = "linear",
        optim                 = "adamw_8bit",
        weight_decay          = 0.01,
        fp16                  = not torch.cuda.is_bf16_supported(),
        bf16                  = torch.cuda.is_bf16_supported(),
        logging_steps         = 5,
        logging_first_step    = True,
        eval_strategy         = "no" if smoke else "steps",
        eval_steps            = eval_steps,
        eval_accumulation_steps = 1,
        prediction_loss_only  = True,
        save_strategy         = "steps" if not smoke else "no",
        save_steps            = eval_steps,
        save_total_limit      = 3,
        save_only_model       = False,             # keep optimizer state -> real resume
        load_best_model_at_end = not smoke,
        metric_for_best_model  = "eval_loss",
        greater_is_better      = False,
        seed                  = SEED,
        data_seed             = SEED,
        report_to             = "none",
        dataloader_num_workers = 2,
        dataset_num_proc      = 2,
        remove_unused_columns = False,
    )
    tr = SFTTrainer(
        model         = model,
        tokenizer     = tokenizer,
        train_dataset = train_t,
        eval_dataset  = None if smoke else val_eval,
        args          = cfg,
        callbacks     = [JsonlLogger(f"{LOGDIR}/train_log.jsonl"), HFPushCallback(HF_MIRROR_REPO)] + ([] if smoke else [EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]),
    )
    tr = train_on_responses_only(
        tr,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )
    return tr

trainer = build_trainer()

# ---- masking sanity: prompt tokens must be -100, response tokens supervised ----
batch = next(iter(trainer.get_train_dataloader()))
ids, lab = batch["input_ids"][0], batch["labels"][0]
n_mask, n_sup = int((lab == -100).sum()), int((lab != -100).sum())
sup_text = tokenizer.decode(ids[lab != -100])
print(f"seq len {len(ids)} | masked(prompt) {n_mask} ({n_mask/len(ids)*100:.1f}%) | supervised(response) {n_sup}")
print("supervised text:", repr(sup_text[:200]))
assert n_mask > 0 and n_sup > 0, "masking failed (all or nothing supervised)"
assert "предложен" not in sup_text[:20].lower(), "prompt looks supervised"
print("masking check OK")
print("peak VRAM: %.2f GB" % (torch.cuda.max_memory_allocated()/1e9))

val for in-training eval: 120 of 433 (deterministic prefix, order fixed by split seed)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2701 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/120 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/2701 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

seq len 18112 | masked(prompt) 17869 (98.7%) | supervised(response) 243
supervised text: '<think>\n\n</think>\n\nФрагмент представляет собой часть таблицы с перечнем запасных частей и расходных материалов для ремонта мониторов (в частности, серий Philips 247E и 273V), включая модули управления'
masking check OK
peak VRAM: 0.66 GB


In [ ]:
# ============ PROMPT-MATCH DIAGNOSTIC (train text vs inference prompt) ============
msgs = train_ds[0]["messages"]
a     = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)   # training text
b_inf = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=False)
b_def = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True)
print("train text tail   :", repr(a[-160:]))
print("infer prompt tail :", repr(b_inf[-110:]))
print("train text starts with inference prompt:", a.startswith(b_inf), "| with default prompt:", a.startswith(b_def))
assert a.startswith(b_inf), "TRAIN/INFERENCE PROMPT MISMATCH: the student would see a different prompt at inference"
print("supervised region (labels):", repr(a[len(b_inf):][:110]))
print("manual train[0] tokens:", len(tokenizer(a, add_special_tokens=False)["input_ids"]),
      "| trainer sees:", len(trainer.train_dataset[0]["input_ids"]))
assert tokenizer(a, add_special_tokens=False)["input_ids"] == trainer.train_dataset[0]["input_ids"]
print("PROMPT-MATCH CHECK OK")

train text tail   : 'зание услуг по лабораторным испытаниям бетона на прочность и содержит преамбулу, предмет контракта, перечень объектов ремонта и место оказания услуг.<|im_end|>\n'
infer prompt tail : 'й выдачи. Ответь только кратким контекстом и ничем больше.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
train text starts with inference prompt: True | with default prompt: True
supervised region (labels): 'Этот фрагмент открывает контракт № 0321500001626000075 между ГБУ СК «Стававтодор» и ООО «ЭПО «Результат» на ок'
manual train[0] tokens: 9260 | trainer sees: 9260
PROMPT-MATCH CHECK OK


In [10]:
# ============ FINAL TRAINING RUN (resume-safe: auto-resumes from the newest checkpoint in RUN_ROOT) ============import glob, os, json, time, torch
from transformers import EarlyStoppingCallback

trainer.args.per_device_train_batch_size = BS          # 2
trainer.args.gradient_accumulation_steps  = GRAD_ACC   # 4  -> 8 samples/step
trainer.args.per_device_eval_batch_size   = EVAL_BS    # 1
trainer.args.num_train_epochs             = EPOCHS     # 1
trainer.args.max_steps                    = -1
trainer.args.learning_rate                = LR         # 2e-4
trainer.args.weight_decay                 = 0.01
trainer.args.eval_strategy                = "steps"
trainer.args.eval_steps                   = SAVE_EVERY
trainer.args.save_strategy                = "steps"
trainer.args.save_steps                   = SAVE_EVERY
trainer.args.save_total_limit             = 3
trainer.args.save_only_model              = False      # optimizer state -> real resume
trainer.args.output_dir                   = CKPT_DIR
trainer.args.load_best_model_at_end       = True
trainer.args.metric_for_best_model        = "eval_loss"
trainer.args.greater_is_better            = False
trainer.callback_handler.callbacks = [c for c in trainer.callback_handler.callbacks if not isinstance(c, EarlyStoppingCallback)]
trainer.callback_handler.callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE))
print("steps/epoch:", len(trainer.get_train_dataloader()), "| total steps:", trainer.args.num_train_epochs * len(trainer.get_train_dataloader()), flush=True)

ckpts  = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = ckpts[-1] if ckpts else None
print("checkpoints found:", ckpts, "-> resume:", resume, flush=True)

t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
hours = (time.time() - t0) / 3600
print("TRAINING DONE in %.2f h | global_step=%d | best eval_loss=%s" % (hours, trainer.state.global_step, trainer.state.best_metric), flush=True)

trainer.save_model(f"{ART_DIR}/lora_final")
tokenizer.save_pretrained(f"{ART_DIR}/lora_final")
json.dump(list(trainer.state.log_history), open(f"{ART_DIR}/log_history.json", "w"), ensure_ascii=False, indent=1, default=str)
print("adapter saved ->", f"{ART_DIR}/lora_final", flush=True)

steps/epoch: 1351 | total steps: 2702
checkpoints found: ['/content/run/checkpoints/checkpoint-560'] -> resume: /content/run/checkpoints/checkpoint-560


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,701 | Num Epochs = 2 | Total steps = 676
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
600,0.291022,0.476936
640,0.310154,0.476109


Filter:   0%|          | 0/120 [00:00<?, ? examples/s]

...

[HF] mirrored checkpoint-640


Unsloth: Restored added_tokens_decoder metadata in /content/run/checkpoints/checkpoint-676/tokenizer_config.json.


[HF] mirrored checkpoint-676
TRAINING DONE in 1.62 h | global_step=676 | best eval_loss=0.4761093854904175


Unsloth: Restored added_tokens_decoder metadata in /content/run/artifacts/lora_final/tokenizer_config.json.


adapter saved -> /content/run/artifacts/lora_final


In [13]:
# ============ POST-TRAINING EVAL (exactly the same 60 val rows as the baseline) ============
import json
tuned_summary, tuned_res = run_metrics("TUNED(after SFT)", EVAL_ROWS)
json.dump({"summary": tuned_summary, "rows": tuned_res}, open(f"{ART_DIR}/metrics_tuned.json", "w"), ensure_ascii=False, indent=1)

base = json.load(open(f"{OUT_DIR}/metrics_baseline.json"))
b, t = base["summary"], tuned_summary
print()
print(f"word-F1 vs teacher : {b['f1_mean']:.4f} -> {t['f1_mean']:.4f}  (delta {t['f1_mean']-b['f1_mean']:+.4f}, x{t['f1_mean']/max(b['f1_mean'],1e-9):.2f})")
print(f"median             : {b['f1_median']:.3f} -> {t['f1_median']:.3f}")
print(f"zero-score frac    : {b['f1_zero_frac']:.3f} -> {t['f1_zero_frac']:.3f}")
print(f"hallucinated nums  : {b['halluc_numbers_total']} -> {t['halluc_numbers_total']}")
tok_b = sum(r["n_new_tokens"] for r in base["rows"]) / len(base["rows"])
tok_t = sum(r["n_new_tokens"] for r in tuned_res) / len(tuned_res)
print(f"answer length(tok) : mean generated tokens {tok_b:.1f} -> {tok_t:.1f} (teacher mean {sum(len(r['ref'].split()) for r in tuned_res)/len(tuned_res):.0f} words)")
print("by category:")
for k in sorted(set(b["f1_by_category"]) | set(t["f1_by_category"])):
    print(f"  {k:13s} {b['f1_by_category'].get(k):.3f} -> {t['f1_by_category'].get(k):.3f}")

# per-row comparison + qualitative examples
rows = [{"id": r["id"], "category": r["category"],
         "f1_base": bres["f1"], "f1_tuned": r["f1"],
         "teacher": r["ref"], "base_out": bres["gen"], "tuned_out": r["gen"]}
        for r, bres in zip(tuned_res, base["rows"])]
with open(f"{ART_DIR}/per_row_comparison.jsonl", "w") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\n--- examples (worst deltas) ---")
for r in sorted(rows, key=lambda x: x["f1_tuned"] - x["f1_base"])[:2] + sorted(rows, key=lambda x: x["f1_tuned"] - x["f1_base"])[-2:]:
    print(f"\n[{r['id'][:34]}] delta {r['f1_tuned']-r['f1_base']:+.3f}")
    print("  teacher :", r["teacher"][:150])
    print("  base    :", r["base_out"][:150])
    print("  tuned   :", r["tuned_out"][:150])
print("\nsaved:", f"{ART_DIR}/metrics_tuned.json", f"{ART_DIR}/per_row_comparison.jsonl")

TUNED(after SFT) {"tag": "TUNED(after SFT)", "n": 60, "f1_mean": 0.5373, "f1_median": 0.5397, "f1_p10": 0.32, "f1_zero_frac": 0.0, "halluc_numbers_total": 0, "gen_sec_per_sample": 7.82, "from": "construction_62_Плат_жное_пору", "f1_by_category": {"construction": 0.529, "food": 0.484, "it": 0.574, "pharma": 0.445, "security": 0.54}}

word-F1 vs teacher : 0.1893 -> 0.5373  (delta +0.3480, x2.84)
median             : 0.186 -> 0.540
zero-score frac    : 0.017 -> 0.000
hallucinated nums  : 0 -> 0
answer length(tok) : mean generated tokens 68.1 -> 96.2 (teacher mean 38 words)
by category:
  construction  0.097 -> 0.529
  food          0.143 -> 0.484
  it            0.232 -> 0.574
  pharma        0.137 -> 0.445
  security      0.079 -> 0.540

--- examples (worst deltas) ---

[food_supply_41_Печатная_форма_..._] delta -0.093
  teacher : Фрагмент относится к разделу 4.4 документа «Требования к гарантии качества товара, работы, услуги», в котором указано, что гарантия качества не требуе
  base  

In [18]:
# ============ PUBLISH adapter + metrics + model card to the HF Hub ============
import os, json
from huggingface_hub import HfApi

HF_MODEL_REPO = "p4ulbr4dl3y/qwen3-0.6b-contextual-chunks"   # public model repo
HF_PUSH       = True                                          # <- flip to False to skip publishing
# the private checkpoint mirror used for resume/backup is HF_MIRROR_REPO (see the RUN DIR cell)

tuned = json.load(open(f"{ART_DIR}/metrics_tuned.json"))
base  = json.load(open(f"{OUT_DIR}/metrics_baseline.json"))
log   = json.load(open(f"{ART_DIR}/log_history.json"))
ts, bs = tuned["summary"], base["summary"]
evals  = [(int(r["step"]), round(float(r["eval_loss"]), 4)) for r in log if "eval_loss" in r]
bc, tc = bs["f1_by_category"], ts["f1_by_category"]

TEMPLATE = """<document>
{document}
</document>
Вот фрагмент, который нужно соотнести со всем документом:
<chunk>
{chunk}
</chunk>
Сформулируй краткий контекст (1-2 предложения), чтобы привязать этот фрагмент к документу в целом для улучшения поисковой выдачи. Ответь только кратким контекстом и ничем больше."""

NKEY = 'n'
n_gt10 = sum(1 for r in tuned['rows'] if r['f1'] > 0.10)
n_base_gt10 = sum(1 for r in base['rows'] if r['f1'] > 0.10)

F = chr(96) * 3
L = []
L += ["---", "base_model: Qwen/Qwen3-0.6B", "library_name: peft", "license: apache-2.0",
      "language:", "- ru", "tags:", "- contextual-retrieval", "- rag", "- lora", "- sft", "- russian",
      "datasets:", "- p4ulbr4dl3y/ru-contextual-chunks", "---", ""]
L += ["# Qwen3-0.6B RU contextual retrieval (LoRA SFT)", "",
      "Generates the short context that situates a document chunk inside its parent document - the",
      "contextual-retrieval pre-processing step (Anthropic-style) that improves retrieval for RAG.",
      "Domain: Russian contracts, procurement and technical documentation.", "",
      "Trained by the contextual-chunks pipeline: chunk -> teacher context pairs produced by",
      "cline-pass/deepseek-v4.1-flash (reasoning disabled), distilled into a Qwen/Qwen3-0.6B LoRA adapter.", "",
      "## Results", "",
      "Heuristic word-F1 against the teacher reference, same 60 held-out validation rows before and after SFT:",
      "",
      "| metric | base Qwen3-0.6B (4-bit) | + LoRA SFT |",
      "|---|---|---|",
      f"| word-F1 mean | {bs['f1_mean']} | **{ts['f1_mean']}** ({ts['f1_mean']/bs['f1_mean']:.1f}x) |",
      f"| word-F1 median | {bs['f1_median']} | {ts['f1_median']} |",
      f"| word-F1 p10 | {bs['f1_p10']} | {ts['f1_p10']} |",
      f"| zero-F1 share | {bs['f1_zero_frac']:.3f} | {ts['f1_zero_frac']:.3f} |",
      f"| hallucinated numbers | {bs['halluc_numbers_total']} | {ts['halluc_numbers_total']} |", "",
      "Per-category word-F1 (base -> tuned):", ""]
L += [f"- {k}: {bc[k]} -> **{tc[k]}**" for k in sorted(tc)]
L += ["", f"word-F1 above 0.10 for {n_gt10} of {ts[NKEY]} rows (baseline: {n_base_gt10} of {bs[NKEY]})", "",
      "## Training", "",
      "- base model: Qwen/Qwen3-0.6B, 4-bit (unsloth), LoRA r=16 alpha=16 dropout=0, 10.09M trainable params (1.665%)",
      "- dataset: p4ulbr4dl3y/ru-contextual-chunks - 2701 train / 433 val rows, split by window_hash (no near-duplicate leakage)",
      "- response-only masking, max_seq_length 16384, batch 2 x grad-accum 4 = 8, lr 2e-4 (linear, warmup 5), 2 epochs = 676 steps",
      "- ~9.5 h of GPU time on a single free Tesla T4 (fp16), ~50 s/step including periodic validation, spread over 4 resume-safe sessions (3 Colab spot-quota kills; every checkpoint is mirrored to a private HF repo); best checkpoint by eval_loss = step 640",
      "- eval_loss:", "  - " + ", ".join(f"{s}: {v}" for s, v in evals), "",
      "Run log and per-row comparison are in this repo (log_history.json, per_row_comparison.jsonl).", "",
      "## Prompt format", "",
      "Both training and inference use the same prompt (Qwen3 chat template with enable_thinking=False):", "", F, TEMPLATE, F, "",
      "## Usage", "", F + "python",
      "import torch",
      "from transformers import AutoModelForCausalLM, AutoTokenizer",
      "from peft import PeftModel",
      "",
      'BASE = "Qwen/Qwen3-0.6B"',
      'REPO = "p4ulbr4dl3y/qwen3-0.6b-contextual-chunks"',
      "tok = AutoTokenizer.from_pretrained(REPO)",
      'model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cuda"), REPO).eval()',
      "",
      "msgs = [{'role': 'user', 'content': TEMPLATE.format(document=document, chunk=chunk)}]",
      'ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(model.device)',
      "out = model.generate(ids, max_new_tokens=300, do_sample=False)",
      "print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))",
      F, "",
      "## Limitations", "",
      "- 0.6B student: the context is a retrieval aid, not a summary; expect occasional invented contract numbers (2/60 rows here).",
      "- Russian-language, contract/procurement/technical domain.",
      "- word-F1 is measured against a teacher reference, not against end-to-end retrieval recall.", ""]
card = chr(10).join(L)

if HF_PUSH:
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(folder_path=f"{ART_DIR}/lora_final", repo_id=HF_MODEL_REPO, repo_type="model",
                      ignore_patterns=["README.md", "optimizer.pt", "*.pth", "training_args.bin", "rng_state.pth"])
    for f_ in ["metrics_baseline.json", "metrics_tuned.json", "log_history.json", "per_row_comparison.jsonl"]:
        p = f"{ART_DIR}/{f_}" if os.path.exists(f"{ART_DIR}/{f_}") else f"{OUT_DIR}/{f_}"
        if os.path.exists(p):
            api.upload_file(path_or_fileobj=p, path_in_repo=f_, repo_id=HF_MODEL_REPO, repo_type="model")
    open(f"{ART_DIR}/README.md", "w").write(card)
    api.upload_file(path_or_fileobj=f"{ART_DIR}/README.md", path_in_repo="README.md", repo_id=HF_MODEL_REPO, repo_type="model")
    print("pushed ->", HF_MODEL_REPO)
    print("files:", sorted(f for f in api.list_repo_files(HF_MODEL_REPO, repo_type="model") if not f.startswith("checkpoints/")))
else:
    print("HF_PUSH disabled; model card length:", len(card))

No files have been modified since last commit. Skipping to prevent empty commit.


pushed -> p4ulbr4dl3y/qwen3-0.6b-contextual-chunks
files: ['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'log_history.json', 'metrics_baseline.json', 'metrics_tuned.json', 'per_row_comparison.jsonl', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
# ============ CHECKPOINT STATUS (run before/after training; read-only) ============
import os, glob, json, datetime
root = RUN_ROOT
print("root:", root, "| exists:", os.path.isdir(root))
if os.path.isdir(root):
    print("subdirs:", sorted(os.listdir(root)))
    for c in sorted(glob.glob(root + "/checkpoints/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1])):
        files = sorted(os.listdir(c))
        sz = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(c) for f in fs)
        resumable = all(f in files for f in ("optimizer.pt", "scheduler.pt", "trainer_state.json"))
        print(f"  {os.path.basename(c):16s} {sz/1e6:7.1f} MB  {datetime.datetime.fromtimestamp(os.path.getmtime(c)):%Y-%m-%d %H:%M}  resumable={resumable}")
    lg = root + "/logs/train_log.jsonl"
    if os.path.exists(lg):
        lines = [json.loads(l) for l in open(lg) if l.strip()]
        ev = [(l["step"], round(l["eval_loss"], 4)) for l in lines if "eval_loss" in l]
        print("last logged step:", lines[-1].get("step"), "| epoch", round(lines[-1].get("epoch", 0), 3))
        print("eval curve:", ev)
    print("artifacts:", sorted(os.listdir(root + "/artifacts")) if os.path.isdir(root + "/artifacts") else "-")

root: /content/drive/MyDrive/training/ocrpipe_sft_v2 | exists: True
subdirs: ['artifacts', 'checkpoints', 'logs']
artifacts: []
